## Feature Engineering
### 01.Initial Setup and Data Loading
Before engineering features, the data must be chronologically sorted. Time-series models cannot look into the future, so sorting prevents accidental look-ahead bias when using pandas .rolling() or .shift() functions.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load Data
df = pd.read_csv("E:/fourth_sem/nifty_ml_hybrid/datasets/raw/nifty_vix_daily.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

### 02.Multi-Horizon Log Returns
Explanation: Financial prices are non-stationary (they drift upwards infinitely). Machine learning models require stationary data. We use Log Returns instead of simple percentage returns because log returns are time-additive ($LogRet_{A \to C} = LogRet_{A \to B} + LogRet_{B \to C}$) and structurally closer to a normal distribution. We calculate this across multiple horizons (1 to 21 days) to give the deep learning models a sense of short, medium, and long-term momentum

In [3]:
# Calculate log returns for 1, 2, 3, 5, 10, and 21 days
for d in [1, 2, 3, 5, 10, 21]:
    df[f'log_ret_{d}d'] = np.log(df['close'] / df['close'].shift(d))

### 03.Market Microstructure(Candle Anatomy)
Explanation: Tree-based models like XGBoost cannot inherently "see" the shape of a daily trading session. A day that opens low and closes high (bullish engulfing) looks identical to a flat day if you only provide the closing price. These features extract the exact intraday behavior—measuring the size of the wicks (hl_pct), the body (body_size), and overnight gaps (gap_pct).

In [4]:
# High-Low spread normalized by previous close
df['hl_pct'] = (df['high'] - df['low']) / df['close'].shift(1)

# Open-Close spread (Intraday momentum)
df['oc_pct'] = (df['close'] - df['open']) / df['open']

# Overnight gap (Opening price vs Previous Close)
df['gap_pct'] = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)

# Size of the candle body relative to the entire day's range
df['body_size'] = abs(df['close'] - df['open']) / (df['high'] - df['low'] + 1e-8)

### 04.Volatility Estimators(The VIX Edge)
Explanation: Standard deviation of close-to-close returns underestimates true market volatility. We add Parkinson Volatility, which uses the High and Low prices to capture extreme intraday swings.
For the VIX, absolute VIX values are less meaningful than relative VIX values. A VIX of 20 means nothing on its own; but if the VIX surges from 12 to 20, it signals panic. We calculate a 21-day rolling Z-score to measure these volatility shocks.

In [5]:
# Realized & Parkinson Volatility
for w in [5, 10, 21]:
    # Standard Realized Volatility (Annualized)
    df[f'rvol_{w}d'] = df['log_ret_1d'].rolling(w).std() * np.sqrt(252)
    
    # Parkinson Volatility (Uses High/Low for intraday variance)
    rs = (1.0 / (4.0 * np.log(2.0))) * ((np.log(df['high'] / df['low']))**2)
    df[f'parkinson_{w}d'] = np.sqrt(rs.rolling(w).mean()) * np.sqrt(252)

# VIX Z-Score: How extreme is today's VIX compared to the last month?
df['vix_zscore_21d'] = (df['india_vix'] - df['india_vix'].rolling(21).mean()) / (df['india_vix'].rolling(21).std() + 1e-8)

# Short-term vs Long-term VIX trend
df['vix_ma_ratio_5_21'] = df['india_vix'].rolling(5).mean() / df['india_vix'].rolling(21).mean()

### 05.Technical & Momentum Indicators
Explanation: Raw Moving Averages (like the 200-SMA) are bad features because a 200-SMA of 10,000 in 2018 is vastly different from a 200-SMA of 25,000 in 2025. Instead, we use Price Relative to SMA (percentage distance from the moving average). We also calculate standard bounded oscillators like RSI and MACD, which give tree models clear boundaries for overbought/oversold splits.

In [6]:
# Normalized Moving Averages
for ma in [10, 20, 50, 200]:
    df[f'sma_{ma}'] = df['close'].rolling(ma).mean()
    df[f'price_vs_sma_{ma}'] = df['close'] / df[f'sma_{ma}'] - 1

# MACD (Moving Average Convergence Divergence)
ema_12 = df['close'].ewm(span=12, adjust=False).mean()
ema_26 = df['close'].ewm(span=26, adjust=False).mean()
df['macd'] = ema_12 - ema_26

# 14-day RSI (Relative Strength Index)
delta = df['close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / (loss + 1e-8)
df['rsi_14'] = 100 - (100 / (1 + rs))

# Normalized Average True Range (ATR)
df['tr'] = np.maximum((df['high'] - df['low']), 
                      np.maximum(abs(df['high'] - df['close'].shift(1)), abs(df['low'] - df['close'].shift(1))))
df['atr_14'] = df['tr'].rolling(14).mean() / df['close']

### 06.Rolling Memory & Drawback
Explanation: Drawdown measures how far the market has fallen from its recent high. This serves as a continuous fear indicator. Autocorrelation measures whether the market is currently "trending" (positive autocorrelation) or "mean-reverting" (negative autocorrelation, choppy).

In [7]:
# Rolling 20-day Drawdown
df['drawdown_20d'] = df['close'] / df['close'].rolling(20).max() - 1

# Rolling 20-day Autocorrelation of Returns
def calc_autocorr(x):
    if np.isnan(x).any() or len(x) < 2: return np.nan
    return pd.Series(x).autocorr(lag=1)

df['autocorr_1d_20d'] = df['log_ret_1d'].rolling(20).apply(calc_autocorr, raw=True)

### 07.Cyclic Calender Encoding
Explanation: If you feed a neural network "Monday = 0" and "Friday = 4", the model assumes the distance between Friday and Monday is huge. In reality, they are adjacent trading days. We map temporal data onto a circle using sine and cosine transformations. This ensures the model smoothly transitions across weeks and years.

In [8]:
# Map months (1-12) to a circle
df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12)
df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12)

# Map days of the week (0-4 for trading days) to a circle
df['dow_sin'] = np.sin(2 * np.pi * df['date'].dt.dayofweek / 5)
df['dow_cos'] = np.cos(2 * np.pi * df['date'].dt.dayofweek / 5)

### 08.Regime Detection Features
Explanation: These are the master switches for your Regime-Conditioned Meta-Learner. regime_200MA defines the macro macro trend (Bull=1, Bear=0). The vol_quartile_63d dynamically buckets the market's current volatility into 4 quartiles based on the last 3 months, preventing the model from assuming a 2019 "high vol" day is the same as a 2020 COVID "high vol" day.

In [9]:
# Macro Trend Binary Flag
df['regime_200MA'] = (df['close'] > df['sma_200']).astype(int)

# Dynamic Volatility Quartiles
def calc_quartile(x):
    if np.isnan(x).any(): return np.nan
    bins = np.percentile(x, [25, 50, 75])
    return np.digitize(x[-1], bins)

df['vol_quartile_63d'] = df['rvol_21d'].rolling(63).apply(calc_quartile, raw=True)

### 09.Target Variables(No Lookahead Bias)
Explanation: The most critical step. You must shift the target backward so that row $t$ contains the features of today, but the target of tomorrow. We compute three different targets to allow you to experiment with Regression, Binary Classification, and Multi-class classification in your research.

In [10]:
# The continuous log return for tomorrow (Regression Target)
df['target_ret_1d'] = df['log_ret_1d'].shift(-1)

# The direction of tomorrow (1 if UP, 0 if DOWN)
df['target_dir_1d'] = (df['target_ret_1d'] > 0).astype(int)

# Dynamic 5-class target (Quintiles) - Classifies tomorrow into 'Extreme Down' to 'Extreme Up'
def calc_quintile(x):
    if np.isnan(x).any(): return np.nan
    bins = np.percentile(x, [20, 40, 60, 80])
    return np.digitize(x[-1], bins)

df['target_quintile_1d'] = df['target_ret_1d'].rolling(252).apply(calc_quintile, raw=True)

# Drop the very last row since we cannot know tomorrow's target yet
df = df.dropna(subset=['target_ret_1d'])

### 10. Save the processed features

In [12]:
output_filename = "E:/fourth_sem/nifty_ml_hybrid/datasets/processed/nifty_engineered_features.csv"
df.to_csv(output_filename, index=False)

print(f"Success! Dataset shape: {df.shape}")
print(f"File saved locally as: '{output_filename}'")

Success! Dataset shape: (2685, 49)
File saved locally as: 'E:/fourth_sem/nifty_ml_hybrid/datasets/processed/nifty_engineered_features.csv'
